# Evo-1 v40 — all-suite combined summary, patched

This version handles older folders that do **not** have a rebuilt done CSV.

Fallback order for each selected run folder:
1. Use rebuilt done CSV if present.
2. Otherwise rebuild a done table directly from `results/<mode>/<result_stem>.done.json` using the manifest.

No LIBERO. No server. No raw JSONL rewrite.


In [ ]:
# CELL 00 — mount Drive and configure
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

from pathlib import Path
import json, datetime, os
from collections import Counter
import pandas as pd
import numpy as np

BASE = Path("/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation")
EXP_ID = "v40_linear130_split_ref5ep_windows_cumulative"
SUITES_WANTED = ["libero_spatial", "libero_goal", "libero_object", "libero_10"]

# Optional exact folders. If auto-discovery picks wrong, paste them here.
MANUAL_RUN_ROOTS = [
    # Path("/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_spatial_researchrigor__20260521_134316__626364eb"),
    # Path("/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_goal_object_researchrigor__20260521_134248__c7b2e3d1"),
    # Path("/content/drive/MyDrive/Evo-1-results/w8_dynamic_a8_ablation/v40_linear130_split_ref5ep_windows_cumulative__freshfolder_manifestchecked_fixed_episode_seed_libero10_researchrigor__20260517_055715__04dff99a"),
]

OUT_ROOT = BASE / f"_ALL4_COMBINED_SUMMARY_PATCHED_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
OUT_ROOT.mkdir(parents=True, exist_ok=True)

if not BASE.exists():
    raise RuntimeError(f"Missing BASE: {BASE}")

print("BASE:", BASE)
print("OUT_ROOT:", OUT_ROOT)


In [ ]:
# CELL 01 — helpers: discover folders + load done records robustly

def manifest_path_for(root):
    p = root / "summaries" / f"manifest_{EXP_ID}.json"
    return p if p.exists() else None

def best_done_csv(root):
    summaries = root / "summaries"
    candidates = [
        summaries / "REBUILT_DONE_RECORDS_AFTER_ALL_MODE_FIX.csv",
        summaries / "REBUILT_DONE_RECORDS_FROM_ALL_MODES_ONLY.csv",
        summaries / "FAST_REBUILD_DONE_RECORDS.csv",
        summaries / "REBUILT_DONE_RECORDS_FROM_RAW.csv",
        summaries / "REPAIR_CLEAN_DONE_RECORDS.csv",
        summaries / "REBUILT_DONE_RECORDS.csv",
        summaries / "REBUILT_DONE_RECORDS_FROM_ALL_MODES.csv",
        summaries / "REBUILT_DONE_RECORDS_FROM_RAW.csv",
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def load_manifest(root):
    mp = manifest_path_for(root)
    if mp is None:
        raise RuntimeError(f"Missing manifest in {root}")
    manifest = json.loads(mp.read_text())
    return manifest

def done_path_for(root, row):
    return root / "results" / str(row["ablation_mode"]) / f'{row["result_stem"]}.done.json'

def load_done_records_from_json(root, manifest):
    rows = []
    missing = []
    bad = []

    for row in manifest:
        p = done_path_for(root, row)
        if not p.exists():
            missing.append({
                "ablation_mode": row.get("ablation_mode"),
                "suite": row.get("suite"),
                "task_id": row.get("task_id"),
                "episode_id": row.get("episode_id"),
                "result_stem": row.get("result_stem"),
                "expected_done_path": str(p),
            })
            continue

        try:
            d = json.loads(p.read_text())
        except Exception as exc:
            bad.append({"path": str(p), "error": repr(exc)})
            continue

        # Canonical identity from manifest, not from possibly stale metadata.
        rec = dict(d)
        rec["ablation_mode"] = row["ablation_mode"]
        rec["suite"] = row["suite"]
        rec["task_id"] = int(row["task_id"])
        rec["episode_id"] = int(row["episode_id"])
        rec["result_stem"] = row["result_stem"]
        rec["done_path"] = str(p)

        # Normalize fields.
        rec["success"] = bool(rec.get("success", False))
        rec["task_fail"] = bool(rec.get("task_fail", not rec["success"]))
        rec["server_crash"] = bool(rec.get("server_crash", rec.get("crash", False)))
        rec["timeout_or_missing"] = bool(rec.get("timeout_or_missing", False))
        rec["client_exception"] = bool(rec.get("client_exception", False))
        rec["egl_cleanup_warning"] = bool(rec.get("egl_cleanup_warning", False))
        rec["log_exists"] = bool(rec.get("log_exists", True))
        rec["jsonl_key_exists"] = bool(rec.get("jsonl_key_exists", True))
        rows.append(rec)

    done = pd.DataFrame(rows)
    missing_df = pd.DataFrame(missing)
    bad_df = pd.DataFrame(bad)

    return done, missing_df, bad_df

def load_done_records(root):
    root = Path(root)
    manifest = load_manifest(root)
    done_csv = best_done_csv(root)

    if done_csv is not None:
        done = pd.read_csv(done_csv)
        done["done_source"] = str(done_csv)
        missing_df = pd.DataFrame()
        bad_df = pd.DataFrame()
        return done, missing_df, bad_df, done_csv

    done, missing_df, bad_df = load_done_records_from_json(root, manifest)
    done["done_source"] = "active_done_json_fallback"
    return done, missing_df, bad_df, None

def folder_info(root):
    root = Path(root)
    mp = manifest_path_for(root)
    if mp is None:
        return None

    try:
        manifest = json.loads(mp.read_text())
    except Exception:
        return None

    suites = sorted(set(str(r.get("suite")) for r in manifest if r.get("suite") is not None))
    modes = sorted(set(str(r.get("ablation_mode")) for r in manifest if r.get("ablation_mode") is not None))
    done_csv = best_done_csv(root)

    done_rows = None
    done_source = None
    try:
        done_tmp, missing_df, bad_df, _ = load_done_records(root)
        done_rows = len(done_tmp)
        done_source = "csv" if done_csv else "done_json_fallback"
    except Exception as exc:
        done_source = "load_failed:" + repr(exc)

    summaries = root / "summaries"

    return {
        "root": root,
        "manifest": mp,
        "suites": suites,
        "modes": modes,
        "manifest_rows": len(manifest),
        "done_csv": done_csv,
        "done_rows": done_rows,
        "done_source": done_source,
        "has_success": (summaries / "MAIN_SUCCESS_BY_MODE.csv").exists(),
        "has_suite_success": (summaries / "MAIN_SUCCESS_BY_SUITE_MODE.csv").exists(),
        "has_request_drift": (summaries / "REQUEST_DRIFT_BY_MODE.csv").exists(),
        "has_flow": (summaries / "FOUR_RANGE_FLOW_SENSITIVITY_BY_MODE.csv").exists(),
        "has_site": (summaries / "SITE_LAYER_TYPE_A8_ERROR_SUMMARY.csv").exists(),
        "mtime": root.stat().st_mtime,
    }

print("CELL_01_HELPERS_READY")


In [ ]:
# CELL 02 — discover and select run folders

infos = []

for r in MANUAL_RUN_ROOTS:
    info = folder_info(Path(r))
    if info:
        infos.append(info)

for r in sorted(BASE.glob(f"{EXP_ID}__*")):
    if r.is_dir():
        info = folder_info(r)
        if info:
            infos.append(info)

# Deduplicate by path.
dedup = {}
for info in infos:
    dedup[str(info["root"])] = info
infos = list(dedup.values())

disc = pd.DataFrame([
    {
        "root": str(i["root"]),
        "suites": ",".join(i["suites"]),
        "manifest_rows": i["manifest_rows"],
        "done_rows": i["done_rows"],
        "done_source": i["done_source"],
        "has_request_drift": i["has_request_drift"],
        "has_flow": i["has_flow"],
        "has_site": i["has_site"],
        "mtime": i["mtime"],
    }
    for i in infos
]).sort_values("mtime", ascending=False)

disc.to_csv(OUT_ROOT / "DISCOVERED_RUN_FOLDERS.csv", index=False)

print("DISCOVERED_RUN_FOLDERS:")
print(disc.to_string(index=False))

wanted = set(SUITES_WANTED)
covered = set()
selected = []

# Prefer folders that cover uncovered suites, have done rows, and are recent.
for info in sorted(
    infos,
    key=lambda i: (
        len(set(i["suites"]) & (wanted - covered)),
        i["done_rows"] or 0,
        bool(i["has_request_drift"]),
        i["mtime"],
    ),
    reverse=True,
):
    suites = set(info["suites"]) & wanted
    if not suites:
        continue
    new_suites = suites - covered
    if not new_suites:
        continue

    selected.append(info)
    covered |= suites

    if covered >= wanted:
        break

sel = pd.DataFrame([
    {
        "root": str(i["root"]),
        "suites": ",".join(i["suites"]),
        "manifest_rows": i["manifest_rows"],
        "done_rows": i["done_rows"],
        "done_source": i["done_source"],
        "done_csv": str(i["done_csv"]) if i["done_csv"] else "",
        "has_request_drift": i["has_request_drift"],
    }
    for i in selected
])

sel.to_csv(OUT_ROOT / "SELECTED_RUN_FOLDERS.csv", index=False)

print("\nSELECTED_RUN_FOLDERS:")
print(sel.to_string(index=False))
print("COVERED_SUITES:", sorted(covered))

missing = wanted - covered
if missing:
    raise RuntimeError(f"Missing suites from selected folders: {sorted(missing)}")


In [ ]:
# CELL 03 — load selected data

def load_csv(path):
    path = Path(path)
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

all_done = []
all_req = []
all_flow = []
all_site = []
missing_done_audits = []
bad_done_audits = []

for info in selected:
    root = Path(info["root"])
    summaries = root / "summaries"
    source = root.name

    done, missing_df, bad_df, done_csv = load_done_records(root)
    done["source_run_root"] = str(root)
    done["source_run_name"] = source
    all_done.append(done)

    if len(missing_df):
        missing_df["source_run_root"] = str(root)
        missing_done_audits.append(missing_df)
    if len(bad_df):
        bad_df["source_run_root"] = str(root)
        bad_done_audits.append(bad_df)

    for fname, bucket in [
        ("REQUEST_DRIFT_BY_MODE.csv", all_req),
        ("FOUR_RANGE_FLOW_SENSITIVITY_BY_MODE.csv", all_flow),
        ("SITE_LAYER_TYPE_A8_ERROR_SUMMARY.csv", all_site),
    ]:
        df = load_csv(summaries / fname)
        if len(df):
            df["source_run_root"] = str(root)
            df["source_run_name"] = source
            bucket.append(df)

done_all = pd.concat(all_done, ignore_index=True)
done_all = done_all[done_all["suite"].isin(SUITES_WANTED)].copy()

key_cols = ["ablation_mode", "suite", "task_id", "episode_id"]
dup_count = int(done_all.duplicated(key_cols).sum())
if dup_count:
    print("WARNING overlapping selected folders; keeping first duplicate rows:", dup_count)
    done_all = done_all.drop_duplicates(key_cols, keep="first").copy()

req_all = pd.concat(all_req, ignore_index=True) if all_req else pd.DataFrame()
flow_all = pd.concat(all_flow, ignore_index=True) if all_flow else pd.DataFrame()
site_all = pd.concat(all_site, ignore_index=True) if all_site else pd.DataFrame()

done_all.to_csv(OUT_ROOT / "ALL4_DONE_RECORDS.csv", index=False)

if missing_done_audits:
    pd.concat(missing_done_audits, ignore_index=True).to_csv(OUT_ROOT / "MISSING_DONE_AUDIT.csv", index=False)
if bad_done_audits:
    pd.concat(bad_done_audits, ignore_index=True).to_csv(OUT_ROOT / "BAD_DONE_AUDIT.csv", index=False)

print("DONE_ROWS:", len(done_all))
print("UNIQUE_KEYS:", done_all[key_cols].drop_duplicates().shape[0])
print("SUITES:", sorted(done_all["suite"].unique()))
print("MODES:", sorted(done_all["ablation_mode"].unique()))
print("REQUEST_DRIFT_ROWS:", len(req_all))
print("FLOW_ROWS:", len(flow_all))
print("SITE_ROWS:", len(site_all))


In [ ]:
# CELL 04 — combined success/failure summaries

done_all["success"] = done_all["success"].astype(bool)

if "log_exists" in done_all.columns:
    done_all["log_exists"] = done_all["log_exists"].astype(bool)
else:
    done_all["log_exists"] = True

if "jsonl_key_exists" in done_all.columns:
    done_all["jsonl_key_exists"] = done_all["jsonl_key_exists"].astype(bool)
else:
    done_all["jsonl_key_exists"] = True

by_mode = done_all.groupby("ablation_mode").agg(
    eval_runs=("success", "count"),
    success=("success", "sum"),
    log_missing=("log_exists", lambda x: int((~x).sum())),
    jsonl_missing=("jsonl_key_exists", lambda x: int((~x).sum())),
).reset_index()

by_mode["task_fail"] = by_mode["eval_runs"] - by_mode["success"]
by_mode["success_rate"] = by_mode["success"] / by_mode["eval_runs"].clip(lower=1)

by_suite_mode = done_all.groupby(["suite", "ablation_mode"]).agg(
    eval_runs=("success", "count"),
    success=("success", "sum"),
).reset_index()
by_suite_mode["task_fail"] = by_suite_mode["eval_runs"] - by_suite_mode["success"]
by_suite_mode["success_rate"] = by_suite_mode["success"] / by_suite_mode["eval_runs"].clip(lower=1)

failed_exact = done_all[~done_all["success"]].copy()
failed_tasks = failed_exact[["suite", "task_id"]].drop_duplicates().sort_values(["suite", "task_id"]).reset_index(drop=True)
failed_task_mode = failed_exact.groupby(["suite", "task_id", "ablation_mode"]).size().reset_index(name="failed_episodes")

by_mode.to_csv(OUT_ROOT / "ALL4_SUCCESS_BY_MODE.csv", index=False)
by_suite_mode.to_csv(OUT_ROOT / "ALL4_SUCCESS_BY_SUITE_MODE.csv", index=False)
failed_exact.to_csv(OUT_ROOT / "ALL4_FAILED_EPISODES_EXACT_LIST.csv", index=False)
failed_tasks.to_csv(OUT_ROOT / "ALL4_FAILED_TASKS_ANY_MODE.csv", index=False)
failed_task_mode.to_csv(OUT_ROOT / "ALL4_FAILED_TASK_MODE_COUNTS.csv", index=False)

print("ALL4_SUCCESS_BY_MODE:")
print(by_mode.to_string(index=False))
print("\nFAILED_TASKS_ANY_MODE:")
print(failed_tasks.to_string(index=False))


In [ ]:
# CELL 05 — combined drift/range/site summaries

if len(req_all):
    req = req_all.copy()
    if "request_records" not in req.columns:
        req["request_records"] = 1

    rows = []
    for mode, g in req.groupby("ablation_mode"):
        w = pd.to_numeric(g["request_records"], errors="coerce").fillna(0)
        row = {"ablation_mode": mode, "request_records": int(w.sum())}

        for col in ["E_action", "final_rms_mean", "final_rms_p95", "E_grip"]:
            if col in g.columns:
                x = pd.to_numeric(g[col], errors="coerce")
                row[col] = float((x * w).sum() / max(w.sum(), 1))

        if "final_rms_max" in g.columns:
            row["final_rms_max"] = float(pd.to_numeric(g["final_rms_max"], errors="coerce").max())

        if "gripper_flip_count" in g.columns:
            row["gripper_flip_count"] = float(pd.to_numeric(g["gripper_flip_count"], errors="coerce").sum())

        rows.append(row)

    req_combined = pd.DataFrame(rows).sort_values("ablation_mode")
else:
    req_combined = pd.DataFrame()

req_combined.to_csv(OUT_ROOT / "ALL4_REQUEST_DRIFT_BY_MODE.csv", index=False)

if len(flow_all):
    flow_all.to_csv(OUT_ROOT / "ALL4_FLOW_RANGE_RAW.csv", index=False)
    num_cols = [
        c for c in flow_all.columns
        if c not in ["ablation_mode", "range_label", "source_run_root", "source_run_name"]
        and pd.api.types.is_numeric_dtype(flow_all[c])
    ]
    flow_combined = flow_all.groupby(["ablation_mode", "range_label"])[num_cols].mean().reset_index() if num_cols else flow_all.drop_duplicates(["ablation_mode", "range_label"])
else:
    flow_combined = pd.DataFrame()

flow_combined.to_csv(OUT_ROOT / "ALL4_FLOW_RANGE_COMBINED.csv", index=False)

if len(site_all):
    site_all.to_csv(OUT_ROOT / "ALL4_SITE_RAW.csv", index=False)
    num_cols = [
        c for c in site_all.columns
        if c not in ["ablation_mode", "site_type", "source_run_root", "source_run_name"]
        and pd.api.types.is_numeric_dtype(site_all[c])
    ]
    site_combined = site_all.groupby(["ablation_mode", "site_type"])[num_cols].mean().reset_index() if num_cols else site_all.drop_duplicates(["ablation_mode", "site_type"])
else:
    site_combined = pd.DataFrame()

site_combined.to_csv(OUT_ROOT / "ALL4_SITE_LAYER_TYPE_A8_ERROR_SUMMARY.csv", index=False)

print("ALL4_REQUEST_DRIFT_BY_MODE:")
print(req_combined.to_string(index=False) if len(req_combined) else "No request drift files found.")

print("\nFLOW_RANGE_ROWS:", len(flow_combined))
print("SITE_ROWS:", len(site_combined))


In [ ]:
# CELL 06 — recommend stress-test modes

mode_rank = by_mode.copy()

if len(req_combined) and "ablation_mode" in req_combined.columns:
    mode_rank = mode_rank.merge(req_combined, on="ablation_mode", how="left")

required_modes = ["v40_w8a16_reference", "v40_w8a8_ALL_00_31"]

dynamic = mode_rank[~mode_rank["ablation_mode"].isin(required_modes)].copy()
dynamic = dynamic.sort_values(["success_rate", "success"], ascending=False)

recommended = required_modes + [m for m in dynamic["ablation_mode"].head(3).tolist() if m not in required_modes]

mode_rank["recommended_for_stress"] = mode_rank["ablation_mode"].isin(recommended)
mode_rank.to_csv(OUT_ROOT / "ALL4_MODE_RANKING_AND_STRESS_RECOMMENDATION.csv", index=False)

print("MODE_RANKING:")
print(mode_rank.sort_values(["recommended_for_stress", "success_rate"], ascending=[False, False]).to_string(index=False))
print("\nRECOMMENDED_STRESS_MODES:", recommended)


In [ ]:
# CELL 07 — write markdown summary

lines = []
lines.append("# Evo-1 v40 all-suite combined summary\n")
lines.append(f"Generated: {datetime.datetime.now().isoformat(timespec='seconds')}\n")
lines.append(f"Output folder: `{OUT_ROOT}`\n")

lines.append("## Selected run folders\n")
for info in selected:
    lines.append(f"- `{info['root']}` | suites={','.join(info['suites'])} | done_rows={info['done_rows']} | source={info['done_source']}\n")

lines.append("\n## Success by mode\n")
lines.append(by_mode.to_markdown(index=False))

lines.append("\n\n## Success by suite and mode\n")
lines.append(by_suite_mode.to_markdown(index=False))

lines.append("\n\n## Failed tasks\n")
lines.append(failed_tasks.to_markdown(index=False) if len(failed_tasks) else "No failed tasks.")

lines.append("\n\n## Stress-test recommended modes\n")
lines.append(", ".join(recommended))

if len(req_combined):
    lines.append("\n\n## Request/action drift by mode\n")
    lines.append(req_combined.to_markdown(index=False))

md = "\n".join(lines)
(OUT_ROOT / "PAPER_STYLE_ALL4_SUMMARY.md").write_text(md)

print(md)
print("\nSAVED:", OUT_ROOT / "PAPER_STYLE_ALL4_SUMMARY.md")
